<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_02/sankey2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram 2
### Target Group → Target → Land Use


# Import libaries

In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors as pc


# Adgang til Airtable

In [ ]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [ ]:
# henter data fra Airtable via API og gemmer det i en pandas.DataFrame
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [ ]:
# Fetch data
tabel0 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets
tabel2 = fetch_airtable_data("tblTRyuT48bBN24QG")   # Slut-land uses

# Fjern rækker med manglende værdier i Target Group-tabellen
tabel0 = tabel0.dropna(subset=['Target Group', 'Targets'])

# Fjern rækker med manglende værdier i Targets-tabellen
tabel1 = tabel1.dropna(subset=['Land uses'])


In [ ]:
# For t0 (Target Group)
t0 = tabel0.copy()
if 'Targets' in t0.columns:
    t0 = t0.explode('Targets')
t0 = t0.rename(columns={
    'Target Group': 'target_group_name',
    'Targets': 'target_id',           # omdøber eksploderede Targets til target_id
    'id': 'target_group_id'
})

# For t1 (Targets)
t1 = tabel1.copy()
if 'Land uses' in t1.columns:
    t1 = t1.explode('Land uses')
t1 = t1.rename(columns={
    'Target name': 'target_name',
    'id': 'target_id',        # omdøb id
    'Land uses': 'land_use_id'
})

# For t2 (Land Uses)
t2 = tabel2.copy()
t2 = t2.rename(columns={
    'Name': 'land_use_name',
    'id': 'land_use_id'       # omdøb id
})


In [ ]:
# Opret forbindelse
con = duckdb.connect()

# Registrer pandas DataFrames som tabeller i DuckDB
con.register('tabel0', t0)
con.register('tabel1_exp', t1)
con.register('tabel2_exp', t2)


In [ ]:
result = con.sql("""
SELECT
  tg.target_group_name,
  t.target_name,
  lu.land_use_name
FROM t0 tg
JOIN t1 t ON tg.target_id = t.target_id
JOIN t2 lu ON t.land_use_id = lu.land_use_id
""").df()

In [ ]:
def forkort_label(label, max_len=40):
    if isinstance(label, str) and len(label) > max_len:
        return label[:max_len] + '…'
    return label

# Anvend på kolonner i join-resultat (fx result)
result['target_group_name_short'] = result['target_group_name'].apply(forkort_label)
result['land_use_name_short'] = result['land_use_name'].apply(forkort_label)

In [ ]:
# Alle labels
all_labels = pd.concat([
    result['target_group_name_short'],
    result['target_name'],
    result['land_use_name_short']
]).unique().tolist()

# Map label til indeks
label_to_index = {label: i for i, label in enumerate(all_labels)}

# Kilde (source) og mål (target) for links
source = result['target_group_name_short'].map(label_to_index)
target = result['target_name'].map(label_to_index)
value = [1] * len(result)  # Vægt 1 pr række

# Andet led links
source2 = result['target_name'].map(label_to_index)
target2 = result['land_use_name_short'].map(label_to_index)
value2 = [1] * len(result)

# Saml alle links
source_all = pd.concat([source, source2], ignore_index=True)
target_all = pd.concat([target, target2], ignore_index=True)
value_all = pd.concat([pd.Series(value), pd.Series(value2)], ignore_index=True)

In [ ]:
# Opret entydige node-id'er (tekniske ID'er – ikke labels)
result['node_TG'] = 'TG_' + result['target_group_name']
result['node_T'] = 'T_' + result['target_name']
result['node_LU'] = 'LU_' + result['land_use_name']

# Saml alle tekniske node-ID'er i rækkefølge og uden gentagelser
all_node_ids = pd.concat([result['node_TG'], result['node_T'], result['node_LU']]).drop_duplicates().tolist()
node_to_index = {node_id: i for i, node_id in enumerate(all_node_ids)}

# Brug tekniske ID'er til at mappe source/target
source = result['node_TG'].map(node_to_index)
target = result['node_T'].map(node_to_index)
source2 = target
target2 = result['node_LU'].map(node_to_index)

# Saml source/target/value
value = [1] * len(result)
value2 = [1] * len(result)

source_all = pd.concat([source, source2], ignore_index=True)
target_all = pd.concat([target, target2], ignore_index=True)
value_all = pd.Series(value + value2)

# Vis labels – pæne, forkortede versioner
def preprocess_label(label, max_len=40, wrap_len=60):
    if not isinstance(label, str):
        return label
    if len(label) > max_len:
        label = label[:max_len] + '…'
    return '\n'.join([label[i:i+wrap_len] for i in range(0, len(label), wrap_len)])

label_lookup = {
    **dict(zip(result['node_TG'], result['target_group_name'].apply(preprocess_label))),
    **dict(zip(result['node_T'], result['target_name'].apply(preprocess_label))),
    **dict(zip(result['node_LU'], result['land_use_name'].apply(preprocess_label)))
}
# Garanteret én label per node_id – i korrekt rækkefølge
all_labels = [label_lookup[node_id] for node_id in all_node_ids]

# Farver
import plotly.colors as pc
node_colors = pc.qualitative.Plotly
node_colors_list = [node_colors[i % len(node_colors)] for i in range(len(all_labels))]

def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

link_colors = [lighten(node_colors_list[src], factor=0.8) for src in source_all]

# Tegn Sankey-diagrammet
import plotly.graph_objects as go

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=80,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=all_labels,
        color=node_colors_list
    ),
    link=dict(
        source=source_all,
        target=target_all,
        value=value_all,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Target Group → Target → Land Use",
    font_size=12,
    height=6000
)

fig.show()


#Download


In [ ]:
# fig.update_layout(title_text="Target Group → Target → Land Use", font_size=12, height=2000)
# fig.write_html("sankey_diagram.html")

# from google.colab import files
# files.download("sankey_diagram.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>